In [ ]:
import pandas as pd
import os
from google.cloud import bigquery
from dotenv import load_dotenv

# 1. Load the environment variable pointing to your gcp_key.json
load_dotenv('../.env')

# 2. SECURITY FIX: Inject the exact path of the service account key
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../gcp_key.json'

# Initialize the official BigQuery client
client = bigquery.Client()

print("Downloading raw data from BigQuery...")

# 3. Extract the raw data
query = "SELECT * FROM `mktng-performance-dashboard.data.raw`;"

# Execute the query and convert it directly into a Pandas DataFrame
df = client.query(query).to_dataframe()

print("Data downloaded! Calculating KPIs...")

# 4. Calculate Marketing KPIs
# Using fillna(0) to prevent division by zero errors
df['ctr'] = (df['clicks'] / df['impressions']).fillna(0)
df['cpc'] = (df['mark_spent'] / df['clicks']).fillna(0)
df['cpa'] = (df['mark_spent'] / df['orders']).fillna(0)
df['roas'] = (df['revenue'] / df['mark_spent']).fillna(0)

# Show a preview of the calculated metrics
print(df[['campaign_id', 'c_date', 'ctr', 'cpc', 'cpa', 'roas']].head())

# 5. Save the results (Local backup)
df.to_csv('../data/processed/metrics_clean.csv', index=False)

# 6. Load processed data back to the cloud using the official Google client
full_destination_table = 'mktng-performance-dashboard.data.marketing_kpis'

print("\nUploading processed data to Google BigQuery...")

# Tell Google to overwrite the table if it already exists (prevents duplicates)
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE" 
)

# Use the client to upload the DataFrame directly
job = client.load_table_from_dataframe(
    df, 
    full_destination_table, 
    job_config=job_config
)

# Wait for the cloud job to finish
job.result()

print(f"✅ Metrics calculated and table '{full_destination_table}' successfully created in the cloud!")

Descargando datos crudos desde BigQuery...


C:\Users\jbav9\OneDrive\Desktop\Portafolio\Digital Marketing Performance Dashboard\marketing-performance-dashboard\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


¡Datos descargados! Calculando KPIs...
   campaign_id      c_date       ctr        cpc          cpa      roas
0       374754  2021-02-01  0.012438  15.757076  3372.014286  1.655687
1       374754  2021-02-02      0.01   4.645895       4639.7  1.039291
2       374754  2021-02-03   0.01348   21.60249  3810.509804  1.307174
3       374754  2021-02-04      0.01  12.447178  3928.394737  1.504177
4       374754  2021-02-05  0.006967   2.428294    4553.9625  1.312483

Subiendo datos procesados a Google BigQuery...
✅ ¡Métricas calculadas y tabla 'mktng-performance-dashboard.data.marketing_kpis' creada en la nube con éxito!
